# testatlasvision from PyPI: end-to-end test
Installs the **published** package (`pip install testatlasvision`), with no local files or uploads, and checks it on a GPU.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Add a Colab Secret named `HF_TOKEN` (key icon in the left sidebar) with **Notebook access** on.
   Your Hugging Face account must already have access to https://huggingface.co/NCAIR1/N-ATLaS.

Run the cells top to bottom. If Colab says "Restart session" after the install cell, restart and continue from the next cell.

## 1. Install from PyPI

In [ ]:
# torchao conflicts with transformers/peft in Colab (same step as the original test-atlas-vision notebook)
!pip uninstall -y -q torchao

# PyPI can take a minute to serve a brand-new release; just re-run this cell if it says "no matching distribution".
!pip install -q "testatlasvision[4bit]"

In [ ]:
import sys, importlib.metadata as md, torch, transformers
import testatlasvision

print("testatlasvision", md.version("testatlasvision"))
print("installed at   ", testatlasvision.__file__)          # must be site-packages / dist-packages, NOT /content
print("transformers   ", transformers.__version__)           # <- send this line to your lead / me (for the version pin)
print("torch          ", torch.__version__)
print("GPU            ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: switch the runtime to a GPU!")
assert "/content" not in testatlasvision.__file__, "This is a local copy, not the PyPI package"
assert torch.cuda.is_available(), "No GPU"

## 2. Load the model (the slow part: downloads about 16 GB)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

from testatlasvision import (TestAtlasVision, GenerationConfig, AtlasVisionError,
                             AuthenticationError, UnsupportedLanguageError, ImageLoadError)

model = TestAtlasVision()       # token comes from the HF_TOKEN Colab secret
print(model)
print(model.info())

## 3. One-shot questions

In [ ]:
import testatlasvision, time
IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"   # two cats on a couch
img = testatlasvision.load_image(IMAGE_URL)
display(img.resize((320, int(320 * img.height / img.width))))

t0 = time.time()
a1 = model.ask(img, "How many animals are in this image?", config=GenerationConfig(max_new_tokens=60))
t_ask = time.time() - t0
print("ask     :", a1, f"({t_ask:.1f}s)")
d1 = model.describe(img)
print("describe:", d1)

## 4. Follow-ups with a Session

In [ ]:
s = model.new_session()
q1 = model.ask(img, "What is the main object in this picture?", session=s, config=GenerationConfig(max_new_tokens=50))
q2 = model.ask(img, "What colour is it?", session=s, config=GenerationConfig(max_new_tokens=50))
print("Q1:", q1); print("Q2:", q2); print(len(s.history), "turns stored")
restored = type(s).from_dict(s.to_dict())          # what the API would save and reload
assert restored.history == s.history

## 5. Multilingual chat (translate, answer, translate back)
Answers should now end properly instead of being cut off mid-sentence.

In [ ]:
chat_results = {}
for lang, q in [("ig", "Kedu ihe dị na foto a?"),
                ("yo", "Kí ni ó wà nínú àwòrán yìí?"),
                ("ha", "Me ke cikin wannan hoton?")]:
    t0 = time.time()
    r = model.chat(img, q, lang=lang, config=GenerationConfig(max_new_tokens=150))
    chat_results[lang] = (r, time.time() - t0)
    print(f"[{lang}] {q}  ({chat_results[lang][1]:.1f}s)\n  -> {r.answer}\n  (English: {r.english_question} -> {r.english_answer})\n")

## 6. Text-only and translation

In [ ]:
t1 = model.text("what is positron? Kọwaa ya n'asụsụ Igbo.", config=GenerationConfig(max_new_tokens=120))
t2 = model.translate("Good morning, how are you?", "yo")
print(t1); print(); print(t2)

## 7. Errors

In [ ]:
seen = {}
for name, fn in {
    "language":  lambda: model.chat(img, "hi", lang="fr"),
    "image":     lambda: model.ask("/no/such/file.jpg", "hi"),
    "question":  lambda: model.ask(img, "   "),
}.items():
    try:
        fn(); seen[name] = None
    except (AtlasVisionError, ValueError) as e:
        seen[name] = type(e).__name__
        print(f"{name:9s}-> {type(e).__name__}: {e}")

## 8. Bad Hugging Face token (the check that was buggy before)

In [ ]:
from testatlasvision.auth import ensure_llm_access
try:
    ensure_llm_access("NCAIR1/N-ATLaS", "hf_thisIsAFakeToken")
    seen["bad_token"] = None
    print("unexpectedly OK")
except Exception as e:
    seen["bad_token"] = type(e).__name__
    print(type(e).__name__, "->", e)

## 9. Summary: all checks in one place

In [ ]:
checks = {
    "ask returns text":                   isinstance(a1, str) and len(a1) > 0,
    "describe returns text":              isinstance(d1, str) and len(d1) > 0,
    "session keeps 2 turns":              len(s.history) == 2,
    "chat ig/yo/ha return answers":       all(r.answer.strip() and r.language == l for l, (r, _) in chat_results.items()),
    "chat keeps english_answer":          all(r.english_answer.strip() for r, _ in chat_results.values()),
    "text() works":                       bool(t1.strip()),
    "translate() works":                  bool(t2.strip()),
    "bad language -> UnsupportedLanguage": seen.get("language") == "UnsupportedLanguageError",
    "bad image -> ImageLoadError":        seen.get("image") == "ImageLoadError",
    "empty question -> ValueError":       seen.get("question") == "ValueError",
    "fake HF token -> AuthenticationError": seen.get("bad_token") == "AuthenticationError",
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
print()
print(f"English ask: {t_ask:.1f}s | non-English chat: " + ", ".join(f"{l} {t:.0f}s" for l, (_, t) in chat_results.items()) + "   (timings for the API team)")
print("ALL CHECKS PASSED" if all(checks.values()) else "SOME CHECKS FAILED: copy this output and send it over")